In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

spark = (SparkSession.builder
         .appName("windows")
         .master("spark://spark-master:7077")
         .config("spark.executor.memory", "512m")
         .getOrCreate())

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/27 10:21:07 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
df = (spark.read.format("json")
      .option("multiLine", "true")
      .load("../data/nobel_prizes.json")
     )

df.show()

+----------+--------------------+--------------------+----+
|  category|           laureates|   overallMotivation|year|
+----------+--------------------+--------------------+----+
| chemistry|[{Carolyn, 1015, ...|                null|2022|
| economics|[{Ben, 1021, "for...|                null|2022|
|literature|[{Annie, 1017, "f...|                null|2022|
|     peace|[{Ales, 1018, "Th...|                null|2022|
|   physics|[{Alain, 1012, "f...|                null|2022|
|  medicine|[{Svante, 1011, "...|                null|2022|
| chemistry|[{Benjamin, 1002,...|                null|2021|
| economics|[{David, 1007, "f...|                null|2021|
|literature|[{Abdulrazak, 100...|                null|2021|
|     peace|[{Maria, 1005, "f...|                null|2021|
|   physics|[{Syukuro, 999, "...|"for groundbreaki...|2021|
|  medicine|[{David, 997, "fo...|                null|2021|
| chemistry|[{Emmanuelle, 991...|                null|2020|
| economics|[{Paul, 995, "for...|       

In [8]:
df_flattened = (
    df
    .withColumn("laureates",
                explode(col("laureates")))
    .select(col("category")
            ,col("year")
            ,col("overallMotivation")
            ,col("laureates.id")
            ,col("laureates.firstname")
            ,col("laureates.surname")
            ,col("laureates.share")
            ,col("laureates.motivation"))
    .filter(col("laureates.firstname").isNotNull() &
            col("laureates.surname").isNotNull()))

df_flattened.show()

+----------+----+--------------------+----+----------+-----------+-----+--------------------+
|  category|year|   overallMotivation|  id| firstname|    surname|share|          motivation|
+----------+----+--------------------+----+----------+-----------+-----+--------------------+
| chemistry|2022|                null|1015|   Carolyn|   Bertozzi|    3|"for the developm...|
| chemistry|2022|                null|1016|    Morten|     Meldal|    3|"for the developm...|
| chemistry|2022|                null| 743|     Barry|  Sharpless|    3|"for the developm...|
| economics|2022|                null|1021|       Ben|   Bernanke|    3|"for research on ...|
| economics|2022|                null|1022|   Douglas|    Diamond|    3|"for research on ...|
| economics|2022|                null|1023|    Philip|     Dybvig|    3|"for research on ...|
|literature|2022|                null|1017|     Annie|     Ernaux|    1|"for the courage ...|
|     peace|2022|                null|1018|      Ales|Bialia

In [11]:
def concat(first_name, last_name):
    return first_name + " " + last_name

concat_udf = udf(concat, StringType())

In [12]:
df_flattened = df_flattened.withColumn("full_name",
                                       concat_udf(df_flattened["firstname"],
                                                  df_flattened["surname"]))

df_flattened.show()

+----------+----+--------------------+----+----------+-----------+-----+--------------------+-----------------+
|  category|year|   overallMotivation|  id| firstname|    surname|share|          motivation|        full_name|
+----------+----+--------------------+----+----------+-----------+-----+--------------------+-----------------+
| chemistry|2022|                null|1015|   Carolyn|   Bertozzi|    3|"for the developm...| Carolyn Bertozzi|
| chemistry|2022|                null|1016|    Morten|     Meldal|    3|"for the developm...|    Morten Meldal|
| chemistry|2022|                null| 743|     Barry|  Sharpless|    3|"for the developm...|  Barry Sharpless|
| economics|2022|                null|1021|       Ben|   Bernanke|    3|"for research on ...|     Ben Bernanke|
| economics|2022|                null|1022|   Douglas|    Diamond|    3|"for research on ...|  Douglas Diamond|
| economics|2022|                null|1023|    Philip|     Dybvig|    3|"for research on ...|    Philip 

In [16]:
def square_udf(x):
    return x ** 2

spark.udf.register("square", square_udf, IntegerType())
spark.udf.register("parseInt", int, IntegerType())

df_flattened.createOrReplaceTempView("prizes")
result = spark.sql("SELECT share, square(parseInt(share)) AS square_share FROM prizes")

result.show()

+-----+------------+
|share|square_share|
+-----+------------+
|    3|           9|
|    3|           9|
|    3|           9|
|    3|           9|
|    3|           9|
|    3|           9|
|    1|           1|
|    3|           9|
|    3|           9|
|    3|           9|
|    1|           1|
|    2|           4|
|    2|           4|
|    2|           4|
|    4|          16|
|    4|          16|
|    1|           1|
|    2|           4|
|    2|           4|
|    4|          16|
+-----+------------+
only showing top 20 rows



25/05/27 11:17:15 WARN SimpleFunctionRegistry: The function square replaced a previously registered function.


In [17]:
spark.stop()